# Extract data from CERCA raw files

In [ ]:
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [5]:
interest_centers = ['CTFC']
file_path = '../data/external/5_Bibliometria_SIRIS_031125/'

## Extract DOI list

**Each file has a different format so we go center by center**

### CTFC

In [6]:
center_name = 'CTFC/'
file_name = 'CTFC_articles_CERCA'

In [7]:
df_CTFC = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx')
df_CTFC['DOI'] = df_CTFC['DOI'].str.replace(r'^https://doi\.org/', '', regex=True)
df_CTFC = df_CTFC[['DOI']]
df_CTFC['Center'] = 'CTFC'
df_CTFC

,DOI,Center
0,10.1038/d41586-022-01955-x,CTFC
1,10.1038/s41586-023-06440-7,CTFC
2,10.1126/science.abo3856,CTFC
3,10.1038/s41586-021-03292-x,CTFC
4,10.1038/s41558-023-01726-0,CTFC
...,...,...
518,10.1098/rspb.2023.2383,CTFC
519,10.1016/j.ecoinf.2024.102557,CTFC
520,10.17660/ActaHortic.2024.1400.31,CTFC
521,10.3390/agronomy14010006,CTFC


In [8]:
df_centers = df_CTFC
df_centers

,DOI,Center
0,10.1038/d41586-022-01955-x,CTFC
1,10.1038/s41586-023-06440-7,CTFC
2,10.1126/science.abo3856,CTFC
3,10.1038/s41586-021-03292-x,CTFC
4,10.1038/s41558-023-01726-0,CTFC
...,...,...
518,10.1098/rspb.2023.2383,CTFC
519,10.1016/j.ecoinf.2024.102557,CTFC
520,10.17660/ActaHortic.2024.1400.31,CTFC
521,10.3390/agronomy14010006,CTFC


## Check which publications are not in OA using DOI

In [9]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [10]:
in_query = str(tuple(df_centers.DOI.str.lower().tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """
df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1002/rra.3996,Jo Halvard Halleraker,9,middle,True,204778367,NO
1,10.1002/rra.3996,Jo Halvard Halleraker,9,middle,True,2800020546,NO
2,10.1177/20552076231177146,Laura Teresa Cabrera-Rivera,4,middle,False,158818660,PR
3,10.1016/j.rser.2023.113729,Jo Halvard Halleraker,13,middle,False,204778367,NO
4,10.1073/pnas.2115329119,Timothy J. Kileen,81,middle,False,4210087861,BO
...,...,...,...,...,...,...,...
9330,10.1016/j.fgb.2024.103937,Michael J. Wingfield,6,middle,False,69552723,ZA
9331,10.1038/s41597-024-03159-6,Greg G. Forsyth,21,middle,False,26092322,ZA
9332,10.1126/science.abo3856,Katherine Bunney,26,middle,False,69552723,ZA
9333,10.1038/s41559-022-01831-x,Graham Durrheim,88,middle,False,1336432201,ZA


In [11]:
df_OA.DOI.nunique()

514

### Identify CERCA authors using OA
- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation [2k affiliations]
  - String search with keywords for = 1 doi per raw affilation [4k affilations]

In [12]:
cerca_centers = {'CTFC' : ['4210117018']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'CTFC' : ['15766328', '123044942', '4210146327']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1002/rra.3996,Jo Halvard Halleraker,9,middle,True,204778367,NO
2,10.1177/20552076231177146,Laura Teresa Cabrera-Rivera,4,middle,False,158818660,PR
3,10.1016/j.rser.2023.113729,Jo Halvard Halleraker,13,middle,False,204778367,NO
4,10.1073/pnas.2115329119,Timothy J. Kileen,81,middle,False,4210087861,BO
5,10.1002/eap.2309,Michaela Plein,17,middle,False,4210164208,LU
...,...,...,...,...,...,...,...
8267,10.3390/fire6120457,João P. Honrado,3,middle,False,182534213,PT
8294,10.1016/j.biocon.2021.109043,Ana Filipa Filipe,2,last,False,141596103,PT
8846,10.1016/j.scitotenv.2022.157980,Adrián Pascual,1,first,True,66946132,US
8897,10.1038/s43247-023-00881-8,Marie Pelagie Elimbi Moudio,3,middle,False,95457486,US


In [13]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1126/science.abo3856,Louise A. Ashton,"School of Biological Sciences, The University ...",889458895,HK
1,10.3390/rs15102627,Willie Doaemo,"Department of Civil Engineering, Papua New Gui...",119444370,PG
2,10.3390/rs15102627,Willie Doaemo,"Morobe Development Foundation, Doyle Street, T...",119444370,PG
3,10.1002/rra.3996,Mauro Carolli,Energy Systems SINTEF Energy Research Trondhei...,4210142273,NO
4,10.1038/s41477-023-01543-5,David Kenfack,"CTFS-ForestGEO, Smithsonian Tropical Research ...",128212801,PA
...,...,...,...,...,...
5720,10.1007/s40725-022-00163-0,Felipe Balocchi,Forestry and Agricultural Biotechnology Instit...,69552723,ZA
5721,10.1111/ecog.06974,Irene Barnes,"Department of Biochemistry, Genetics and Micro...",69552723,ZA
5722,10.1111/1365-2745.14098,Cang Hui,"Theoretical Ecology Unit, African Institute fo...",26092322,ZA
5723,10.1111/1365-2745.14098,Cang Hui,"Department of Mathematical Sciences, Centre fo...",26092322,ZA


In [14]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
to_check.to_csv('to_check_CTFC.csv')
to_check

238


,DOI
raw_affiliation,
"Department of Crop and Forest Sciences, University of Lleida, Lleida, Spain",14
"Forest Science and Technology Centre of Catalonia (CTFC), Solsona, Spain",14
"Department of Agricultural and Forest Sciences and Engineering, University of Lleida, Lleida, Spain",13
"Department of Crop and Forest Sciences, University of Lleida, 25198 Lleida, Spain",7
"LINCGlobal, Museo Nacional de Ciencias Naturales, CSIC, Madrid, Spain",6
...,...
"Centre Tecnològic Forestal de Catalunya (CTFC), Solsona, Lleida, Spain",2
"CTFC, Solsona, Spain",2
"University of Zaragoza - Universidad de Zaragoza [Zaragoza] (Pedro Cerbuna,12\r\n50009 Zaragoza - España - Spain)",2


In [35]:
df_check = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', '>1CTFC', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.7284894837476099

In [16]:
cerca_string = {'CTFC' : ['ctfc']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_check_2 = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_check_2['raw_affiliation'] = df_check_2['raw_affiliation'].str.lower()

df_check_2.to_csv('to_check_CTFC_1.csv')
df_check_2

# df_check_2['CERCA'] = df_check_2['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False).map({True: 'TRUE', False: 'FALSE'})
# df_check_2

,raw_affiliation,DOI
0,) forest science and technology centre of cata...,1
1,university of barcelona (gran via de les corts...,1
2,"university pablo de olavide (ctra. de utrera, ...",1
3,university institute of research in environmen...,1
4,university autònoma de barcelona bellaterra spain,1
...,...,...
1148,wildlife conservation medicine research group ...,1
1149,wildlife conservation medicine research group ...,1
1150,) department of agricultural and forest scienc...,1
1151,"university of eastern finland, department of g...",1


In [36]:
df_check_2 = g2d.download('1J7uywXX7fsxjNbUSi24uQJiW5bTjbKHba1RjqdYBwQA', '=1CTFC', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.7839388145315488

In [37]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]
# check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_2[df_check_2.CERCA == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.8355640535372849

In [38]:
df_check = pd.concat((cerca_authors, check_1, check_2))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
3913,10.1080/10095020.2024.2429376,Antoni Trasobares,3,middle,False,4210117018,ES,NaN
3914,10.1080/22797254.2024.2425337,Antoni Trasobares,4,last,False,4210117018,ES,NaN
3915,10.1038/s41586-021-03292-x,Antoni Trasobares,30,middle,False,4210117018,ES,NaN
3916,10.1007/s11676-024-01735-5,Antoni Trasobares,5,last,False,4210117018,ES,NaN
3920,10.3832/ifor4376-016,Antoni Trasobares,3,middle,False,4210117018,ES,NaN
...,...,...,...,...,...,...,...,...
4282,10.1016/j.jsames.2023.104625,Gemma Piqué,<NA>,NaN,<NA>,4210117018,ES,Centre de Ciència I Tecnologia Forestal de Cat...
4287,10.23818/limn.43.02,Frederic Casals,<NA>,NaN,<NA>,4210117018,ES,Forest Science and Technology Centre of Catalo...
4291,10.1080/00218839.2023.2277988,Gérard Bota,<NA>,NaN,<NA>,4210117018,ES,"Conservation Biology Group, Landscape Dynamics..."
4292,10.1017/s0959270922000430,Gérard Bota,<NA>,NaN,<NA>,4210117018,ES,"Conservation Biology Group, Landspcape Dynamic..."


In [39]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_CTFC.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1002/rra.3996,Jo Halvard Halleraker,9,middle,True,204778367,NO,False,CTFC
1,10.1002/rra.3996,Jo Halvard Halleraker,9,middle,True,2800020546,NO,False,CTFC
2,10.1177/20552076231177146,Laura Teresa Cabrera-Rivera,4,middle,False,158818660,PR,False,CTFC
3,10.1016/j.rser.2023.113729,Jo Halvard Halleraker,13,middle,False,204778367,NO,False,CTFC
4,10.1073/pnas.2115329119,Timothy J. Kileen,81,middle,False,4210087861,BO,False,CTFC
...,...,...,...,...,...,...,...,...,...
9098,10.1016/j.fgb.2024.103937,Michael J. Wingfield,6,middle,False,69552723,ZA,False,CTFC
9099,10.1038/s41597-024-03159-6,Greg G. Forsyth,21,middle,False,26092322,ZA,False,CTFC
9100,10.1126/science.abo3856,Katherine Bunney,26,middle,False,69552723,ZA,False,CTFC
9101,10.1038/s41559-022-01831-x,Graham Durrheim,88,middle,False,1336432201,ZA,False,CTFC


In [40]:
df_final[df_final.CERCA == True].DOI.nunique() / df_centers.DOI.nunique()

0.8126195028680688